In [8]:
import numpy as np
import matplotlib.pyplot as plt

class SmartSource:
    def __init__(self, Y=10000.0):
        self.Y = Y

    def generate_photons(self, E, position=(0, 0, 0), batch_size=100):
        """
        Генератор фотонов. 
        Выдает направления порциями по batch_size штук.
        """
        # 1. Случайное число фотонов по Пуассону со средним E*Y
        mean_photons = E * self.Y
        N_photons = np.random.poisson(mean_photons)
        
        # Генерируем все направления сразу (изотропно)
        cos_theta = np.random.uniform(-1, 1, N_photons)
        phi = np.random.uniform(0, 2 * np.pi, N_photons)
        sin_theta = np.sqrt(1 - cos_theta**2)
        
        nx = sin_theta * np.cos(phi)
        ny = sin_theta * np.sin(phi)
        nz = cos_theta
        
        directions = np.column_stack((nx, ny, nz))
        
        # 2. Возвращаем порциями по batch_size (yield)
        for i in range(0, N_photons, batch_size):
            yield np.array(position), directions[i:i+batch_size]

# --- Проверка ---
src = SmartSource()

# Смотрим, сколько фотонов сгенерировалось для 1 МэВ
batch_count = 0
total_photons = 0
for pos, dirs in src.generate_photons(E=1.0, batch_size=100):
    batch_count += 1
    total_photons += len(dirs)
    
print(f"Сгенерировано порций (батчей): {batch_count}")
print(f"Всего фотонов (должно быть ~10000): {total_photons}")

Сгенерировано порций (батчей): 100
Всего фотонов (должно быть ~10000): 9969


In [9]:
# Для начала скопируем классы Detector и Source из ДЗ 1 (без метода is_hit, он нам тут не нужен)
class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R; self.N = N; self.r = r; self.a = a
        indices = np.arange(0, N, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices / N)
        theta = np.pi * (1 + 5**0.5) * indices
        x = np.cos(theta) * np.sin(phi) * R
        y = np.sin(theta) * np.sin(phi) * R
        z = np.cos(phi) * R
        self.pmt_positions = np.column_stack((x, y, z))

def find_hit_pmt(detector, start_pos, directions):
    R = detector.R
    b = 2 * np.sum(start_pos * directions, axis=1)
    c = np.sum(start_pos**2) - R**2
    discriminant = b**2 - 4 * c
    t = (-b + np.sqrt(discriminant)) / 2
    hit_points = start_pos + t[:, np.newaxis] * directions
    
    # --- НОВОЕ: ЗАТУХАНИЕ ---
    # Путь, который пролетел фотон
    path_length = t
    attenuation_length = 8.0  # метров
    # Вероятность выжить = exp(-L / L_att)
    survival_prob = np.exp(-path_length / attenuation_length)
    # Бросаем кубик для каждого фотона
    survived = np.random.uniform(0, 1, len(path_length)) < survival_prob
    
    # Если фотон не выжил, ставим ему hit_point в 100000 (чтобы не попал ни в один ФЭУ)
    hit_points[~survived] = 100000.0 
    # -------------------------
    
    hit_indices = np.full(len(hit_points), -1, dtype=int)
    for i, pt in enumerate(hit_points):
        distances = np.linalg.norm(detector.pmt_positions - pt, axis=1)
        if np.min(distances) <= detector.r:
            hit_indices[i] = np.argmin(distances)
            
    return hit_indices

# Функция для симуляции одного события (с затуханием)
def simulate_event_atten(det, src, E, position):
    total_pe = 0
    for pos, dirs in src.generate_photons(E, position):
        hits = find_hit_pmt(det, pos, dirs)
        for pmt_id in range(det.N):
            photons_in_pmt = np.sum(hits == pmt_id)
            total_pe += np.random.poisson(det.a * photons_in_pmt)
    return total_pe

# Проверка для центра и смещенного источника
det = Detector()
src = SmartSource()

# Центр (0,0,0)
charges_center = [simulate_event_atten(det, src, E=1.0, position=(0,0,0)) for _ in range(100)]
# Смещен на 1.5 м вверх (внутри сферы, так как R=2)
charges_shifted = [simulate_event_atten(det, src, E=1.0, position=(0,0,1.5)) for _ in range(100)]

print(f"Средний заряд (центр, с затуханием): {np.mean(charges_center):.1f}")
print(f"Средний заряд (смещен на 1.5 м, с затуханием): {np.mean(charges_shifted):.1f}")

Средний заряд (центр, с затуханием): 96.9
Средний заряд (смещен на 1.5 м, с затуханием): 102.4


In [10]:
import time

# 1. Медленный способ (цикл по каждому фотону)
def slow_way(N):
    cos_theta = np.zeros(N)
    phi = np.zeros(N)
    for i in range(N):
        cos_theta[i] = np.random.uniform(-1, 1)
        phi[i] = np.random.uniform(0, 2*np.pi)
    
    directions = np.zeros((N, 3))
    for i in range(N):
        st = np.sqrt(1 - cos_theta[i]**2)
        directions[i, 0] = st * np.cos(phi[i])
        directions[i, 1] = st * np.sin(phi[i])
        directions[i, 2] = cos_theta[i]
    return directions

# 2. Быстрый способ (numpy)
def fast_way(N):
    cos_theta = np.random.uniform(-1, 1, N)
    phi = np.random.uniform(0, 2*np.pi, N)
    st = np.sqrt(1 - cos_theta**2)
    
    directions = np.zeros((N, 3))
    directions[:, 0] = st * np.cos(phi)
    directions[:, 1] = st * np.sin(phi)
    directions[:, 2] = cos_theta
    return directions

N_photons = 100_000

start = time.time()
slow_way(N_photons)
time_slow = time.time() - start

start = time.time()
fast_way(N_photons)
time_fast = time.time() - start

print(f"Медленный способ (циклы for): {time_slow:.4f} сек")
print(f"Быстрый способ (numpy): {time_fast:.4f} сек")
print(f"Выигрыш в скорости: {time_slow / time_fast:.1f} раз!")

Медленный способ (циклы for): 0.7741 сек
Быстрый способ (numpy): 0.0079 сек
Выигрыш в скорости: 97.6 раз!


In [11]:
from scipy.spatial import cKDTree
import time

class BigDetector(Detector):
    def __init__(self):
        # Вызываем конструктор родителя с новыми параметрами
        super().__init__(R=15.0, N=2000, r=0.1, a=0.5)
        # Создаем KD-дерево для быстрого поиска
        self.tree = cKDTree(self.pmt_positions)

    def is_hit_fast(self, points):
        """Быстрый поиск с использованием KDTree"""
        # Находим расстояние до ближайшего ФЭУ для каждой точки
        distances, indices = self.tree.query(points)
        # Попадание, если расстояние меньше радиуса
        return distances <= self.r

# Проверка
big_det = BigDetector()
print(f"Создан большой детектор: R={big_det.R}, N={big_det.N}")

# Генерируем случайные точки на сфере (для теста)
test_points = np.random.randn(10000, 3)
test_points = test_points / np.linalg.norm(test_points, axis=1)[:, np.newaxis] * big_det.R

# Замеряем время быстрого поиска
start = time.time()
hits = big_det.is_hit_fast(test_points)
end = time.time()
print(f"Быстрый поиск (KDTree) для 10000 точек: {end - start:.4f} сек")
print(f"Из них попало в ФЭУ: {np.sum(hits)}")

Создан большой детектор: R=15.0, N=2000
Быстрый поиск (KDTree) для 10000 точек: 0.0070 сек
Из них попало в ФЭУ: 229
